# Maia Lite 0.3.0 — Ensaio integrado: modelo + sampler + checkpoint (v1)

**Objetivo:** comparar 10 updates contínuos contra 5 updates + checkpoint + restauração + 5 updates. O modelo é o MaiaGPT 507M original. **Os batches são sintéticos**, gerados a partir de IDs determinísticos de blocos, sem ler tokens do Corpus v8. O catálogo SQLite é consultado apenas para identidade/estrutura.

**Custo:** 15 updates de treinamento A100 80 GB (~2–5 minutos mais inicialização, checkpoint e comparações). Exige RAM/Drive suficientes para checkpoints de ~5,7 GiB e dois modelos em sequência. O notebook não usa checkpoints anteriores nem escreve no corpus. Não é um teste de throughput de produção.


In [ ]:
import os, json, math, time, random, hashlib, sqlite3, gc, bisect, copy
from pathlib import Path
from dataclasses import asdict
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from dataclasses import dataclass
from datetime import datetime, timezone
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
except ImportError:
    pass
assert torch.cuda.is_available(), 'Requer GPU CUDA (A100 80GB recomendada)'
assert torch.cuda.get_device_properties(0).total_memory >= 70*2**30, 'Selecione A100 80GB'
assert torch.cuda.is_bf16_supported(), 'BF16 indisponível'
print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__)
SEED=20261105
PROJECT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
RECOVERY=PROJECT/'corpus_v8/runs/acquisition_v8_5_001/pretraining_production_v1_full/recovery_audit_v4'
DB=RECOVERY/'state_operational_v5.sqlite3'
assert DB.is_file(), f'SQLite ausente: {DB}'
RUN_DIR=PROJECT/'pretraining_05b_integrated_resume_synthetic_v1'
RUN_DIR.mkdir(parents=True,exist_ok=True)
# Determinismo rigoroso pode reduzir desempenho; falhar em vez de ignorar operações não determinísticas.
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
seed_all(SEED)


## 1. Catálogo somente leitura e identidade congelada

A identidade do ensaio incorpora um resumo estável das linhas `shard` e o SHA-256 do próprio arquivo SQLite. O SQLite **não é usado como autorização de treinamento**. Nenhum payload real é lido neste ensaio.


In [ ]:
def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for buf in iter(lambda:f.read(8*1024*1024),b''):h.update(buf)
    return h.hexdigest()
con=sqlite3.connect(DB.resolve().as_uri()+'?mode=ro',uri=True)
try:
    assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
    records=con.execute('SELECT source,split,shard_index,path,blocks,sha256 FROM shard ORDER BY source,split,shard_index').fetchall()
finally:con.close()
assert len(records)==60612, f'Inventário alterado: {len(records)}'
assert all(int(r[4])>0 and len(r[5])==64 for r in records)
train=[{'source':r[0],'split':r[1],'index':r[2],'path':r[3],'blocks':int(r[4]),'sha256':r[5]} for r in records if r[1]=='train']
assert sum(s['blocks'] for s in train)==3390434
canonical=json.dumps(records,ensure_ascii=False,separators=(',',':')).encode()
CATALOG_HASH=hashlib.sha256(canonical).hexdigest()
print('Shards:',len(records),'Train blocks:',sum(s['blocks'] for s in train))
print('Catalog fingerprint:',CATALOG_HASH)
print('READ-ONLY CATALOG: PASS')


Shards: 60612 Train blocks: 3390434
Catalog fingerprint: 737afb495abd1f1b1947c85fc85255615e53473002032bfb904537a4cee54985
READ-ONLY CATALOG: PASS


## 2. Modelo MaiaGPT original (507.026.432 parâmetros)


In [4]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss
model=MaiaGPT(cfg)
params=sum(p.numel() for p in model.parameters())
print('Parameters:',format(params,','),'(~%.3f B)'%(params/1e9))
assert 480_000_000<=params<=530_000_000
assert model.head.weight.data_ptr()==model.tok.weight.data_ptr()


Parameters: 507,026,432 (~0.507 B)


## 3. Sampler indexável e gerador de batches sintéticos

Uma permutação afim bijetiva percorre cada época sem repetir blocos. A posição global determina exatamente o shard e o bloco. Para este ensaio, os **tokens são gerados sinteticamente** a partir do índice global e não são lidos dos arquivos. O alvo da última posição é `-100` para não atravessar a fronteira do bloco.


In [5]:
class SeekableBlockSampler:
    def __init__(self,shards,seed):
        self.shards=shards
        self.starts=[0]
        for s in shards:self.starts.append(self.starts[-1]+s['blocks'])
        self.n=self.starts[-1];self.seed=seed
    def at(self,position):
        epoch,slot=divmod(int(position),self.n)
        rng=random.Random(self.seed+epoch)
        a=rng.randrange(1,self.n)
        while math.gcd(a,self.n)!=1:a=(a+1)%self.n or 1
        b=rng.randrange(self.n)
        ordinal=(a*slot+b)%self.n
        si=bisect.bisect_right(self.starts,ordinal)-1
        s=self.shards[si]
        return (s['source'],s['split'],int(s['index']),ordinal-self.starts[si])
sampler=SeekableBlockSampler(train,SEED)
assert len({sampler.at(i) for i in range(320)})==320
MICROBATCH=8;ACCUM=4;SEQ=2048;BLOCKS_PER_UPDATE=MICROBATCH*ACCUM
WARMUP=2000;LR=3e-4;MIN_LR=3e-5;WD=0.1;CLIP=1.0
TARGET_UPDATES=math.ceil(20*507026432/(MICROBATCH*ACCUM*SEQ))
def lr_at(step):
    if step<WARMUP:return LR*(step+1)/WARMUP
    progress=min(1.,max(0.,(step-WARMUP)/max(1,TARGET_UPDATES-WARMUP)))
    return MIN_LR+0.5*(LR-MIN_LR)*(1+math.cos(math.pi*progress))
def block_tokens(position):
    # Independentemente da ordem de leitura, o mesmo bloco tem os mesmos tokens sintéticos.
    # Não consome o RNG global de treino.
    key=sampler.at(position)
    digest=hashlib.sha256(repr(key).encode()).digest()
    seed=int.from_bytes(digest[:8],'little')%(2**63-1)
    g=torch.Generator(device='cpu').manual_seed(seed)
    return torch.randint(0,cfg.vocab_size,(SEQ,),generator=g,dtype=torch.int64)
def synthetic_batch(position):
    ids=torch.stack([block_tokens(position+i) for i in range(MICROBATCH)])
    x=ids.to('cuda',non_blocking=True)
    y=torch.empty_like(x);y[:,:-1]=x[:,1:];y[:,-1]=-100
    return x,y
print('Sampler unique first 320: PASS')


Sampler unique first 320: PASS


## 4. Checkpoint atômico unificado

Inclui pesos, AdamW, posição do sampler, estado de LR (derivado do update), RNG Python/NumPy/Torch/CUDA e identidade. O `torch.load(weights_only=False)` só deve ser usado para checkpoints **produzidos por este próprio notebook e confiáveis**.


In [8]:
IDENTITY={'run':'maia_05b_integrated_resume_synthetic_v1','mode':'synthetic_only',
          'config':asdict(cfg),'params':params,'seed':SEED,'catalog_sha256':CATALOG_HASH,
          'blocks_per_update':BLOCKS_PER_UPDATE,'seq':SEQ,'lr':LR,'warmup':WARMUP}
CKPT=RUN_DIR/'synthetic_resume_at_5.pt'
REPORT=RUN_DIR/'integrated_resume_report.json'
def init_train():
    seed_all(SEED)
    m=MaiaGPT(cfg).to('cuda')
    opt=torch.optim.AdamW(m.parameters(),lr=LR,betas=(0.9,0.95),eps=1e-8,weight_decay=WD)
    return m,opt

def rng_snapshot():
    return {'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),
            'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all()}
def rng_restore(st):
    random.setstate(st['python_rng']);np.random.set_state(st['numpy_rng'])
    torch.set_rng_state(st['torch_rng']);torch.cuda.set_rng_state_all(st['cuda_rng'])
def save_integrated(m,opt,step):
    state={'identity':IDENTITY,'step':step,'next_position':step*BLOCKS_PER_UPDATE,
           'model':m.state_dict(),'optimizer':opt.state_dict(),'rng':rng_snapshot(),
           'saved_utc':datetime.now(timezone.utc).isoformat()}
    tmp=CKPT.with_suffix('.pt.tmp');torch.save(state,tmp);os.replace(tmp,CKPT)
    print('CHECKPOINT SAVED:',CKPT,'GiB:',round(CKPT.stat().st_size/2**30,2),flush=True)
def load_integrated(m,opt):
    state=torch.load(CKPT,map_location='cpu',weights_only=False)
    assert state['identity']==IDENTITY, 'Identidade incompatível'
    assert state['next_position']==state['step']*BLOCKS_PER_UPDATE
    assert state['step']==5
    m.load_state_dict(state['model']);opt.load_state_dict(state['optimizer'])
    # optimizer.load_state_dict preserva step scalars no device esperado conforme PyTorch.
    rng_restore(state['rng'])
    step=int(state['step']);pos=int(state['next_position'])
    del state;gc.collect()
    return step,pos

def train_steps(m,opt,start,end,log):
    m.train();position=start*BLOCKS_PER_UPDATE
    for step in range(start,end):
        t=time.monotonic()
        opt.zero_grad(set_to_none=True)
        for group in opt.param_groups:group['lr']=lr_at(step)
        positions=[]
        for micro in range(ACCUM):
            batch_pos=position+micro*MICROBATCH
            positions.extend(sampler.at(batch_pos+j) for j in range(MICROBATCH))
            x,y=synthetic_batch(batch_pos)
            with torch.autocast(device_type='cuda',dtype=torch.bfloat16):
                _,loss=m(x,y)
            (loss/ACCUM).backward()
            del x,y
        norm=torch.nn.utils.clip_grad_norm_(m.parameters(),CLIP)
        opt.step()
        position+=BLOCKS_PER_UPDATE
        entry={'step':step+1,'next_position':position,'lr':lr_at(step),
               'loss':float(loss.detach()),'grad_norm':float(norm),'block_keys':positions}
        log.append(entry)
        print(f"update={step+1} loss={entry['loss']:.5f} pos={position} sec={time.monotonic()-t:.1f}",flush=True)
    return position

def fingerprint(m,opt):
    # Hash do estado serializado de cada tensor: detecta diferenças bit a bit.
    h=hashlib.sha256()
    def add_tensor(t):
        a=t.detach().contiguous().cpu()
        h.update(str(a.dtype).encode());h.update(str(tuple(a.shape)).encode())
        h.update(a.reshape(-1).view(torch.uint8).numpy().tobytes())
    for k,v in sorted(m.state_dict().items()):h.update(k.encode());add_tensor(v)
    # Usar IDs de parâmetros ordenados pela ordem dos grupos, não endereços de memória.
    for p in [p for group in opt.param_groups for p in group['params']]:
        state=opt.state.get(p,{})
        for k in sorted(state):
            h.update(k.encode());v=state[k]
            if torch.is_tensor(v):add_tensor(v)
            else:h.update(repr(v).encode())
    return h.hexdigest()


## 5. Execução A: 10 atualizações contínuas

O estado final é reduzido a um fingerprint e o modelo é liberado antes da execução B. Execute **uma única vez** em um runtime limpo.


In [ ]:
assert not CKPT.exists(), 'Checkpoint deste ensaio já existe: use novo RUN_DIR; não sobrescreva resultados anteriores'
assert not REPORT.exists(), 'Relatório já existe: use novo RUN_DIR'
m,opt=init_train();continuous=[]
train_steps(m,opt,0,10,continuous)
fp_cont=fingerprint(m,opt)
print('Continuous fingerprint:',fp_cont)
del m,opt;gc.collect();torch.cuda.empty_cache()


update=1 loss=11.02989 pos=32 sec=2.0
update=2 loss=11.02182 pos=64 sec=2.0
update=3 loss=11.02094 pos=96 sec=2.0
update=4 loss=11.03569 pos=128 sec=2.0


## 6. Execução B: 5 atualizações, checkpoint e retomada

Após salvar o checkpoint, descartamos modelo e otimizador, instanciamos ambos novamente e restauramos o estado completo.


In [ ]:
m,opt=init_train();resumed=[]
train_steps(m,opt,0,5,resumed)
save_integrated(m,opt,5)
del m,opt;gc.collect();torch.cuda.empty_cache()
# Simula um runtime novo no mesmo Colab, sem reutilizar os objetos de modelo/optimizer.
m,opt=init_train()
step,pos=load_integrated(m,opt)
assert (step,pos)==(5,160)
print('RESTORE VERIFIED: step=5 next_position=160')
train_steps(m,opt,step,10,resumed)
fp_resume=fingerprint(m,opt)
print('Resumed fingerprint:',fp_resume)


## 7. Comparação e relatório

Verifica a sequência exata de blocos, posições, LR, losses e fingerprints de pesos/AdamW. Um eventual desacordo bit a bit será registrado como falha de equivalência, não ocultado.


In [ ]:
assert len(continuous)==len(resumed)==10
same_blocks=all(a['block_keys']==b['block_keys'] for a,b in zip(continuous,resumed))
same_positions=all(a['next_position']==b['next_position'] for a,b in zip(continuous,resumed))
same_lr=all(a['lr']==b['lr'] for a,b in zip(continuous,resumed))
loss_diffs=[abs(a['loss']-b['loss']) for a,b in zip(continuous,resumed)]
same_state=(fp_cont==fp_resume)
report={'mode':'synthetic_only','model_params':params,'catalog_sha256':CATALOG_HASH,
        'updates':10,'blocks_per_update':BLOCKS_PER_UPDATE,
        'resume_step':5,'final_next_position':resumed[-1]['next_position'],
        'same_blocks':same_blocks,'same_positions':same_positions,'same_lr':same_lr,
        'max_loss_diff':max(loss_diffs),'same_model_and_adamw_bytes':same_state,
        'continuous_fingerprint':fp_cont,'resumed_fingerprint':fp_resume,
        'passed':bool(same_blocks and same_positions and same_lr and same_state)}
REPORT.write_text(json.dumps(report,indent=2,ensure_ascii=False))
print(json.dumps(report,indent=2,ensure_ascii=False))
print('Report:',REPORT)
assert same_blocks and same_positions and same_lr, 'SAMPLER / LR RESUME FAILED'
if same_state: print('INTEGRATED RESUME VERIFIED: BITWISE MATCH')
else: print('INTEGRATED RESUME: DATA ORDER VERIFIED, NUMERICAL STATE DIFFERS — investigate deterministic kernels')
del m,opt;gc.collect();torch.cuda.empty_cache()


## Interpretação e próximo passo

- **PASS bit a bit:** confirma retomada do modelo, AdamW e sampler **para dados sintéticos** neste ambiente.
- **Mesmos blocos, estados diferentes:** investigar determinismo de kernels, CUDA, RNG e operações BF16.
- **Próxima fase:** leitor real `uint32` com cache local, verificação SHA-256 e manifesto de fontes aprovadas; jamais interpretar a existência de shards como autorização de uso em treinamento.
- **Pré-treinamento definitivo:** inicializar pesos novos e salvar identidade, autorização, tokenizer e posição do dataloader; não reutilizar pesos do ensaio sintético.
